# 10 - Search the bulletins and explain a forecast

We use the Chroma collection from notebook 09 to find bulletin passages related to a question,
and combine them with a forecast to write a short, sourced answer.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
except ImportError:
    pass

In [ ]:
# !pip install -q chromadb sentence-transformers pyyaml

from pathlib import Path
import yaml
import chromadb
from chromadb.utils import embedding_functions

REPO = Path.cwd()
config = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))

client = chromadb.PersistentClient(path=str(REPO / config["vector_store"]["persist_dir"]))
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=config["embedding_candidates"][0]["id"])
collection = client.get_collection(config["vector_store"]["collection_name"], embedding_function=embedding_fn)

print(collection.count(), "chunks available to search")

## A simple search function

Given a question, return the `k` most similar chunks. `before_date` is optional: when we give
it a date, we only get chunks published on or before that date. This matters for a forecast -
we should only use bulletins that existed at the time the forecast was made, not ones from
later that would give the answer away.

In [ ]:
def search(query, k=5, before_date=None):
    # before_date is a "YYYY-MM-DD" string; Chroma can only compare numbers,
    # so we turn it into the same YYYYMMDD number used when the index was built.
    where = None
    if before_date:
        where = {"published_number": {"$lte": int(before_date.replace("-", ""))}}
    result = collection.query(query_texts=[query], n_results=k, where=where,
                              include=["documents", "metadatas", "distances"])
    hits = []
    for doc, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        hits.append({"text": doc, "similarity": round(1 - distance, 3), **meta})
    return hits

In [ ]:
for hit in search("Kuinka monta uutta avointa työpaikkaa ilmoitettiin?", k=3):
    print(f"({hit['similarity']}) {hit['title']}")
    print(" ", hit["text"][:180])

Chroma can only filter numbers, not date strings directly, which is why `search` converts `before_date` into a plain number first. Let's check the filter actually excludes later bulletins.

In [ ]:
recent = search("avoimet työpaikat", k=5, before_date="2020-01-01")
for hit in recent:
    assert hit["published"] <= "2020-01-01", hit
print("all", len(recent), "results are from before 2020")

# a document with no known date (the Statistics Finland releases) has no
# business appearing under an old cutoff either
early_hits = search("job vacancies", k=10, before_date="2015-01-01")
assert all(hit["source"] != "statfin" for hit in early_hits)

## A very small "forecast explainer"

This is where the forecasting model and the RAG meet. The forecast numbers below are a stand-in
(the real ones will come from the fine-tuned model); the point here is just to show how a
question, a forecast, and the retrieved bulletin text fit together into one answer.

In [ ]:
example_forecast = {
    "series": "Uusimaa, all occupations",
    "origin_quarter": "2025Q4",
    "target_quarter": "2026Q4",
    "latest_value": 36400,
    "predicted_value": 33500,
}

def explain_forecast(question, forecast, before_date=None):
    hits = search(question, k=2, before_date=before_date)
    change = "fall" if forecast["predicted_value"] < forecast["latest_value"] else "rise"
    print(f"Forecast: vacancies for {forecast['series']} are expected to {change} "
          f"from {forecast['latest_value']} to {forecast['predicted_value']} "
          f"by {forecast['target_quarter']}.")
    if not hits:
        print("No related bulletin text was found for this question.")
        return
    print("What the bulletins say around that time:")
    for hit in hits:
        print(f"- {hit['title']}: {hit['text'][:200]}")

explain_forecast("Why might vacancies in Uusimaa fall?", example_forecast, before_date="2025-12-31")

## What is still missing

- The forecast above is made up; it should come from the fine-tuned model once it is ready.
- `explain_forecast` just prints the retrieved text - a real version would ask a language model
  to turn it into a proper sentence, still only using what the bulletins actually say.
- Tables in the bulletins (region by region numbers) are not indexed well, since the text
  extraction turns them into loose numbers without their column headers. Regional or
  occupation-specific questions may come back with nothing useful; that is expected for now.